# CURE-Rec advanced reviewer ablations
Run one guarded action at a time. Results are new evidence, not replacements for archived studies.


In [1]:
from pathlib import Path
import sys, pandas as pd
CWD=Path.cwd().resolve()
CANDIDATES=[CWD, CWD/'paper-ideas'/'CURE-Rec'/'code', *CWD.parents]
ROOT=next((p for p in CANDIDATES if (p/'pyproject.toml').exists() and (p/'cure_rec').exists()), None)
if ROOT is None: raise RuntimeError('Open from the CURE-Rec code/notebooks directory or repository root.')
sys.path[:] = [str(ROOT), *[x for x in sys.path if x != str(ROOT)]]
from cure_rec.config import load_settings
from cure_rec.pipeline import run_experiment
from cure_rec.revision_advanced import select_objective, sampled_shapley
FULL_CONFIG=ROOT/'configs'/'curesim_full.yaml'


In [2]:
RUN_OBJECTIVE_ABLATION = True
RUN_SAMPLED_SHAPLEY = False
SEED = 300
assert not (RUN_OBJECTIVE_ABLATION and RUN_SAMPLED_SHAPLEY)


## Action 1 — maximin/mean and hard/penalty ablation


In [3]:
if RUN_OBJECTIVE_ABLATION:
    cfg=load_settings(FULL_CONFIG); cfg.run.seed=SEED; cfg.run.output_root=ROOT/'runs'
    _,game,_=run_experiment(cfg)
    rows=[select_objective(game,cfg,objective=o,constraint_mode=c) for o in ('maximin','mean') for c in ('hard','penalty')]
    display(pd.DataFrame(rows))
else: print('Disabled.')


2026-08-10 22:06:25,824 | INFO | run_started | {"config_hash": "712f7dbdcb27ec7e", "run_id": "curesim-full-20260810T210625Z-d22be6fc"}
2026-08-10 22:06:25,827 | INFO | exact_game_started | {}
2026-08-10 22:06:25,829 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-10 22:13:22,850 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.1406521512316774, "scenario": "nominal", "shapley_efficiency_gap": 0.0}
2026-08-10 22:13:22,852 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-10 22:16:24,670 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.12824569278118791, "scenario": "fatigue_stress", "shapley_efficiency_gap": 0.0}
2026-08-10 22:16:24,700 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-10 22:19:48,472 | INFO | scenario_game_completed | {"grand_coalition_improvem

,mask,objective,constraint_mode,score,feasible,cost,relevance_delta_lower,provider_disparity_upper,fatigue_upper
0,1,maximin,hard,0.289936,True,0.05,-0.0545,0.201319,0.0
1,1,maximin,penalty,0.289936,True,0.05,-0.0545,0.201319,0.0
2,1,mean,hard,0.303462,True,0.05,-0.0545,0.201319,0.0
3,1,mean,penalty,0.303462,True,0.05,-0.0545,0.201319,0.0


## Action 2 — exact versus sampled Shapley


In [4]:
if RUN_SAMPLED_SHAPLEY:
    cfg=load_settings(FULL_CONFIG); cfg.run.seed=SEED; cfg.run.output_root=ROOT/'runs'
    _,game,_=run_experiment(cfg)
    exact=game.robust_shapley
    rows=[]
    for budget in (32,128,512,2048):
        est=sampled_shapley(game.robust_improvements,budget,seed=SEED)
        for player in exact: rows.append({'permutations':budget,'intervention':player,'exact':exact[player],'estimate':est[player],'absolute_error':abs(exact[player]-est[player])})
    display(pd.DataFrame(rows))
else: print('Disabled.')


Disabled.


CRN-off, scaling, user-level bootstrap, and a second dataset require additional simulator/evaluator implementations and should not be substituted with unvalidated shortcuts.
